# Netflix Titles: Data Cleaning
Reads `../data/netflix_titles.csv` and produces three files for the Streamlit dashboard:

| File | Grain | Used for |
|---|---|---|
| `netflix_titles_cleaned.csv` | one row per title | overview, release-year, duration, ratings |
| `genres_long.csv` | one row per title and genre | genre popularity |
| `countries_long.csv` | one row per title and country | regional patterns |

In [ ]:
import pandas as pd
import numpy as np

## 1. Load data and make a working copy

In [ ]:
df = pd.read_csv("../data/netflix_titles.csv")
df_clean = df.copy()
print(f"Raw shape: {df.shape}")

## 2. Checking for duplicates
`show_id` is unique, so a full-row check alone is too weak. Also check on `title + type + release_year`.

In [ ]:
print("Full-row duplicates:          ", df_clean.duplicated().sum())
print("show_id duplicates:           ", df_clean["show_id"].duplicated().sum())
print("title/type/year duplicates:   ", df_clean.duplicated(subset=["title", "type", "release_year"]).sum())

## 3. Checking for null values

In [ ]:
df_clean.isnull().sum().sort_values(ascending=False)

## 4. Strip whitespace in text columns
Done first so later steps (null checks, parsing, splitting) are not affected by stray spaces.

In [ ]:
text_columns = ["type", "title", "director", "cast", "country", "date_added",
                "rating", "duration", "listed_in", "description"]

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

## 5. Fix rows where the duration landed in the `rating` column
A few raw rows have values like `74 min` in `rating` and an empty `duration`. Instead of dropping them, move the value into `duration` and mark the rating as unknown.

In [ ]:
misplaced = df_clean["duration"].isna() & df_clean["rating"].str.contains("min", na=False)
print("Rows with duration stored in rating:", misplaced.sum())
display(df_clean.loc[misplaced, ["show_id", "title", "rating", "duration"]])

df_clean.loc[misplaced, "duration"] = df_clean.loc[misplaced, "rating"]
df_clean.loc[misplaced, "rating"] = np.nan

## 6. Normalizing null values
Text columns get a consistent `Unknown` label. Rows with no `duration` cannot be used and are dropped (expected to be 0 after step 5).

In [ ]:
for col in ["director", "cast", "country", "rating"]:
    df_clean[col] = df_clean[col].fillna("Unknown")

rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=["duration"])
print("Rows dropped for missing duration:", rows_before - len(df_clean))

## 7. Convert `date_added` to datetime
Rows with a missing `date_added` are **kept** (the release-year and genre analysis does not need it). Their `year_added`, `month_added` and `day_added` stay empty, so filter on `year_added.notna()` in time-based charts.

In [ ]:
df_clean["date_added"] = pd.to_datetime(df_clean["date_added"], format="%B %d, %Y", errors="coerce")

print("Missing date_added:", df_clean["date_added"].isna().sum())
print("dtype:", df_clean["date_added"].dtype)

## 8. Creating new date features

In [ ]:
df_clean["year_added"] = df_clean["date_added"].dt.year.astype("Int64")
df_clean["month_added"] = df_clean["date_added"].dt.month_name()
df_clean["day_added"] = df_clean["date_added"].dt.day_name()

# Years between release and arrival on Netflix
df_clean["release_lag"] = (df_clean["year_added"] - df_clean["release_year"]).astype("Int64")

## 9. Separating movie and TV show durations

In [ ]:
df_clean["duration_value"] = pd.to_numeric(df_clean["duration"].str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")
df_clean["duration_unit"] = df_clean["duration"].str.extract(r"([A-Za-z]+)")[0]

# Convenience columns: minutes for movies, seasons for TV shows
df_clean["movie_minutes"] = df_clean["duration_value"].where(df_clean["duration_unit"] == "min")
df_clean["tv_seasons"] = df_clean["duration_value"].where(df_clean["duration_unit"].isin(["Season", "Seasons"]))

df_clean[["duration", "duration_value", "duration_unit", "movie_minutes", "tv_seasons"]].head()

## 10. Extra features for the dashboard

In [ ]:
# Decade of release
df_clean["release_decade"] = (df_clean["release_year"] // 10 * 10).astype(str) + "s"

# Main producing country (first one listed)
df_clean["primary_country"] = df_clean["country"].str.split(",").str[0].str.strip()
df_clean.loc[df_clean["primary_country"] == "", "primary_country"] = "Unknown"

# Audience tier from the rating
audience_map = {
    "TV-Y": "Kids", "TV-Y7": "Kids", "TV-Y7-FV": "Kids", "TV-G": "Kids", "G": "Kids",
    "TV-PG": "Older Kids", "PG": "Older Kids",
    "TV-14": "Teens", "PG-13": "Teens",
    "TV-MA": "Adults", "R": "Adults", "NC-17": "Adults",
}
df_clean["audience"] = df_clean["rating"].map(audience_map).fillna("Not Rated / Unknown")

df_clean["audience"].value_counts()

## 11. Remove duplicate titles

In [ ]:
dupes = df_clean.duplicated(subset=["title", "type", "release_year"]).sum()
print("Duplicate titles removed:", dupes)
df_clean = df_clean.drop_duplicates(subset=["title", "type", "release_year"], keep="first")

## 12. Split multi-value columns
`listed_in` (genres) and `country` hold comma-separated lists. For genre popularity and regional charts each title needs one row per value.

In [ ]:
def explode_column(df, source_col, new_col):
    out = df[["show_id", "title", "type", "release_year", "year_added", source_col]].copy()
    out[new_col] = out[source_col].str.split(",")
    out = out.explode(new_col)
    out[new_col] = out[new_col].str.strip()
    out = out[out[new_col].notna() & (out[new_col] != "")]
    return out.drop(columns=source_col).reset_index(drop=True)

df_genres = explode_column(df_clean, "listed_in", "genre")
df_countries = explode_column(df_clean, "country", "country_single")

print("Genre rows:  ", len(df_genres), "| distinct genres:", df_genres["genre"].nunique())
print("Country rows:", len(df_countries), "| distinct countries:", df_countries["country_single"].nunique())

## 13. Validation checks

In [ ]:
current_year = pd.Timestamp.now().year

assert df_clean["show_id"].is_unique, "show_id is not unique"
assert df_clean["release_year"].between(1900, current_year).all(), "release_year out of range"
assert df_clean["duration"].notna().all(), "duration still has nulls"
assert df_genres["genre"].notna().all() and df_countries["country_single"].notna().all()

added_before_release = (df_clean["release_lag"] < 0).sum()
print("Titles added to Netflix before their release year (check if large):", added_before_release)
print("Titles with missing date_added:", df_clean["date_added"].isna().sum())
print("Final shape:", df_clean.shape)

## 14. Reset the index and check the final structure

In [ ]:
df_clean = df_clean.reset_index(drop=True)
df_clean.info()

## 15. Saving the datasets

In [ ]:
df_clean.to_csv("../data/netflix_titles_cleaned.csv", index=False)
df_genres.to_csv("../data/genres_long.csv", index=False)
df_countries.to_csv("../data/countries_long.csv", index=False)
print("Saved 3 files to ../data/")